## Transform Payments data
### 1. Extract date and time from payment_timestamp and create new columns paymnet_date and Payment_time
### 2. Map payment_status to contain descriptive values (1- Success, 2-Pending, 3-Cancelled, 4-Failed)
### 3. Write traformed data to silver schema

In [0]:
df_payments = spark.read.table("gizmobox_catalog_noori.bronze.py1_payments")
display(df_payments)

In [0]:
from pyspark.sql.functions import date_format, when
df_payments_casting = df_payments.select(
    df_payments['payment_id'],
    df_payments['order_id'],
    date_format(df_payments['payment_timestamp'], 'yyyy-MM-dd').alias('payment_date'),
    date_format(df_payments['payment_timestamp'], 'hh:mm:ss').alias('payment_time'),
    when(df_payments['palyment_status'].cast('string') == '1', 'Success').when(df_payments['palyment_status'].cast('string') == '2', 'Pending').when(df_payments['palyment_status'].cast('string') == '3', 'Failed').when(df_payments['palyment_status'].cast('string') == '4', 'Refunded').alias('payment_status'),
    df_payments['payment_method']
    
    )
display(df_payments_casting)



In [0]:
%sql
select 
payment_id,
order_id,
cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
date_format(payment_timestamp, 'hh:mm:ss') as payment_time,
case  payment_status when  1 then 'Success'
when  2 then 'Pending'
when  3 then 'Cancelled'
when  4 then 'Failed'
end as payment_status,
payment_menthod

from gizmobox_catalog_noori.bronze.payments

In [0]:
%sql
create table gizmobox_catalog_noori.silver.payments
select 
payment_id,
order_id,
cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
date_format(payment_timestamp, 'hh:mm:ss') as payment_time,
case  payment_status when  1 then 'Success'
when  2 then 'Pending'
when  3 then 'Cancelled'
when  4 then 'Failed'
end as payment_status,
payment_menthod

from gizmobox_catalog_noori.bronze.payments;

In [0]:
%sql
select * from gizmobox_catalog_noori.silver.payments